##Create Gold Fact Sales

In [0]:
from pyspark.sql.functions import col

CATALOG = "capretailnew"
SILVER = CATALOG + ".silver"
GOLD = CATALOG + ".gold"

sales = spark.table(SILVER + ".silver_sales_order_lines")
gold_dim_product = spark.table(GOLD + ".gold_dim_product")
gold_dim_store = spark.table(GOLD + ".gold_dim_store")

# Start with Silver sales
gold_fact_sales = sales

# Calculate gross sales
gold_fact_sales = gold_fact_sales.withColumn(
    "gross_sales",
    (col("quantity") * col("unit_price")).cast("decimal(18,2)")
)

# Calculate discount amount
gold_fact_sales = gold_fact_sales.withColumn(
    "discount_amount",
    (
        col("gross_sales") *
        col("discount_pct") / 100
    ).cast("decimal(18,2)")
)

# Calculate net sales
gold_fact_sales = gold_fact_sales.withColumn(
    "net_sales",
    (
        col("gross_sales") -
        col("discount_amount")
    ).cast("decimal(18,2)")
)

# Add tax
gold_fact_sales = gold_fact_sales.withColumn(
    "sales_with_tax",
    (
        col("net_sales") +
        col("tax_amount")
    ).cast("decimal(18,2)")
)

# Add product information
gold_fact_sales = gold_fact_sales.join(
    gold_dim_product.select(
        "product_id",
        "product_name",
        "category",
        "subcategory"
    ),
    on="product_id",
    how="left"
)

# Add store information
gold_fact_sales = gold_fact_sales.join(
    gold_dim_store.select(
        "store_id",
        "store_name",
        "city",
        "region"
    ),
    on="store_id",
    how="left"
)

# Save Gold fact table
gold_fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD + ".gold_fact_sales")

print("Gold Fact Sales created successfully")

##Check Gold Fact Sales

In [0]:
display(
    spark.table(GOLD + ".gold_fact_sales")
    .select(
        "order_id",
        "order_line_id",
        "product_id",
        "product_name",
        "store_id",
        "quantity",
        "gross_sales",
        "discount_amount",
        "net_sales",
        "tax_amount",
        "sales_with_tax"
    )
    .limit(20)
)

##Check total sales

In [0]:
from pyspark.sql.functions import sum

display(
    gold_fact_sales.agg(
        sum("gross_sales").alias("total_gross_sales"),
        sum("discount_amount").alias("total_discount"),
        sum("net_sales").alias("total_net_sales"),
        sum("tax_amount").alias("total_tax")
    )
)

##Join Returns with Sales and Create Gold Fact Returns

In [0]:
returns = spark.table(SILVER + ".silver_sales_returns")

# Select original sales information
sales_for_returns = sales.select(
    "order_line_id",
    col("order_id").alias("original_order_id"),
    col("order_ts").alias("original_order_ts"),
    col("quantity").alias("original_quantity"),
    col("order_status").alias("original_order_status")
)

# Join returns with sales
gold_fact_returns = returns.join(
    sales_for_returns,
    on="order_line_id",
    how="left"
)

# Add product information
gold_fact_returns = gold_fact_returns.join(
    gold_dim_product.select(
        "product_id",
        "product_name",
        "category"
    ),
    on="product_id",
    how="left"
)

# Add store information
gold_fact_returns = gold_fact_returns.join(
    gold_dim_store.select(
        "store_id",
        "store_name",
        "region"
    ),
    on="store_id",
    how="left"
)

# Save the table
gold_fact_returns.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD + ".gold_fact_returns")

print("Gold Fact Returns created")

##Validate Returns

In [0]:
print(
    "Total Gold Returns:",
    gold_fact_returns.count()
)

display(
    gold_fact_returns.groupBy(
        "sales_link_status"
    ).count()
)

display(
    gold_fact_returns.groupBy(
        "approved_flag"
    ).count()
)

display(
    gold_fact_returns.groupBy(
        "return_reason"
    ).count()
)

##Create Gold Fact Inventory Movements

In [0]:
from pyspark.sql.functions import to_date

inventory = spark.table(SILVER + ".silver_inventory_movements")

# Start with inventory movements
gold_fact_inventory = inventory

# Add product information
gold_fact_inventory = gold_fact_inventory.join(
    gold_dim_product.select(
        "product_id",
        "product_name",
        "category",
        "unit_of_measure"
    ),
    on="product_id",
    how="left"
)

# Add store information
gold_fact_inventory = gold_fact_inventory.join(
    gold_dim_store.select(
        "store_id",
        "store_name",
        "region",
        col("warehouse_id").alias("store_warehouse_id")
    ),
    on="store_id",
    how="left"
)

# Extract movement date
gold_fact_inventory = gold_fact_inventory.withColumn(
    "movement_date",
    to_date("movement_ts")
)

# Save Gold table
gold_fact_inventory.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD + ".gold_fact_inventory_movements")

print("Gold Inventory Fact created")

##Validate inventory movements

In [0]:
from pyspark.sql.functions import *
print(
    "Total Inventory Movements:",
    gold_fact_inventory.count()
)

display(
    gold_fact_inventory.groupBy(
        "movement_type"
    ).agg(
        count("*").alias("total_movements"),
        sum("quantity_change").alias("total_quantity_change")
    )
)